# 01 — Data Cleaning

Source: IBM's Telco Customer Churn dataset — 7,043 customers, 21 fields (demographics,
account/contract details, subscribed services, charges, and churn status).


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 140)

df = pd.read_csv("../data/raw_telco_churn.csv")
print(df.shape)
df.head(3)


(7043, 21)


In [2]:
print(df.dtypes)
print()
print("Duplicate customerIDs:", df["customerID"].duplicated().sum())


customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Duplicate customerIDs: 0


## `TotalCharges` is typed as text — why?

In [3]:
bad_rows = df[pd.to_numeric(df["TotalCharges"], errors="coerce").isna()]
print(f"{len(bad_rows)} rows have a non-numeric TotalCharges value")
print(bad_rows[["tenure", "MonthlyCharges", "TotalCharges"]])


11 rows have a non-numeric TotalCharges value
      tenure  MonthlyCharges TotalCharges
488        0           52.55             
753        0           20.25             
936        0           80.85             
1082       0           25.75             
1340       0           56.05             
3331       0           19.85             
3826       0           25.35             
4380       0           20.00             
5218       0           19.70             
6670       0           73.35             
6754       0           61.90             


All 11 blank `TotalCharges` values belong to customers with `tenure = 0` — brand-new
customers who haven't been billed yet. This isn't corrupted data; it's a legitimate zero, encoded
as blank. Filled with `0` (not dropped, and not filled with a mean, which would misrepresent a
real new-customer state) and the column is converted to numeric.

In [4]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
print(df["TotalCharges"].describe())


count    7043.000000
mean     2279.734304
std      2266.794470
min         0.000000
25%       398.550000
50%      1394.550000
75%      3786.600000
max      8684.800000
Name: TotalCharges, dtype: float64


## Standardizing categorical values

In [5]:
cat_cols = df.select_dtypes(include="object").columns.drop("customerID")
for c in cat_cols:
    print(c, ":", df[c].unique().tolist())


gender : ['Female', 'Male']
Partner : ['Yes', 'No']
Dependents : ['No', 'Yes']
PhoneService : ['No', 'Yes']
MultipleLines : ['No phone service', 'No', 'Yes']
InternetService : ['DSL', 'Fiber optic', 'No']
OnlineSecurity : ['No', 'Yes', 'No internet service']
OnlineBackup : ['Yes', 'No', 'No internet service']
DeviceProtection : ['No', 'Yes', 'No internet service']
TechSupport : ['No', 'Yes', 'No internet service']
StreamingTV : ['No', 'Yes', 'No internet service']
StreamingMovies : ['No', 'Yes', 'No internet service']
Contract : ['Month-to-month', 'One year', 'Two year']
PaperlessBilling : ['Yes', 'No']
PaymentMethod : ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)']
Churn : ['No', 'Yes']


Several service columns (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`,
`StreamingTV`, `StreamingMovies`) use `"No internet service"` as a third category, and
`MultipleLines` uses `"No phone service"`. These are collapsed to `"No"` where the underlying
fact is simply "doesn't have this service" — the *reason* (no internet/phone at all) is already
fully captured by the `InternetService`/`PhoneService` columns, so keeping a third category here
would just be redundant encoding of the same fact under a different name.

In [6]:
collapse_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
for c in collapse_cols:
    df[c] = df[c].replace("No internet service", "No")
df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")

df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

print("Churn class balance:")
print(df["Churn"].value_counts(normalize=True).round(3))


Churn class balance:
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


In [7]:
df.to_csv("../data/cleaned_telco_churn.csv", index=False)
print(f"Exported cleaned_telco_churn.csv: {df.shape}")


Exported cleaned_telco_churn.csv: (7043, 21)
